In [4]:
# 1. Install dependencies
!pip install spacy pandas
!python -m spacy download en_core_web_sm

import pandas as pd
import json
import urllib.request
import spacy
import random
import os

# Create data directory
os.makedirs('/kaggle/working/data', exist_ok=True)

# Load Spacy NLP model
nlp = spacy.load("en_core_web_sm")

print("--- Step 1: Downloading and Saving RAW FOLIO Dataset ---")
url = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl"
raw_file_path = "/kaggle/working/data/folio_raw_original.jsonl"

# Download and save the exact raw file
urllib.request.urlretrieve(url, raw_file_path)
print(f"Raw dataset successfully saved to: {raw_file_path}")

# Load into Pandas DataFrame for processing
data = []
with open(raw_file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
        
df = pd.DataFrame(data)

# Ensure premises are a single string (if they are lists)
if isinstance(df['premises'].iloc[0], list):
    df['premises'] = df['premises'].apply(lambda x: ' '.join(x))

print(f"Loaded {len(df)} samples into DataFrame.\n")

print("--- Step 2: Applying Caroline Filter & Shuffling (H1 Preparation) ---")
# Synthetic nonsense words
SYNTHETIC_NAMES = ["Boroogove", "Tweequ", "Snark", "Jubjub", "Bandersnatch", 
                   "Mome", "Rath", "Jabberwock", "Tumtum", "Gyre"]

def neutralize_text(text):
    """Replaces Named Entities with synthetic words to remove prior knowledge."""
    doc = nlp(text)
    entity_map = {}
    synthetic_pool = SYNTHETIC_NAMES.copy()
    random.shuffle(synthetic_pool)
    
    neutralized_text = text
    # Sort entities by length (longest first) to avoid partial replacements
    entities = sorted([ent.text for ent in doc.ents if ent.label_ in ['PERSON', 'ORG', 'GPE']], key=len, reverse=True)
    
    for ent in entities:
        if ent not in entity_map and synthetic_pool:
            entity_map[ent] = synthetic_pool.pop()
            
    for real_word, synthetic_word in entity_map.items():
        neutralized_text = neutralized_text.replace(real_word, synthetic_word)
        
    return neutralized_text

def shuffle_premises(premises_text):
    """Shuffles the order of the premises."""
    sentences = [s.strip() for s in premises_text.split('\n') if s.strip()]
    if len(sentences) == 1:
        sentences = [s.strip() for s in premises_text.split('.') if s.strip()]
    random.shuffle(sentences)
    return '.\n'.join(sentences) + ('.' if not sentences[-1].endswith('.') else '')

# Apply transformations
df['caroline_premises'] = df['premises'].apply(neutralize_text)
df['caroline_conclusion'] = df['conclusion'].apply(neutralize_text)
df['shuffled_premises'] = df['premises'].apply(shuffle_premises)

# Safely extract columns (assign an explicit ID if none exists)
if 'example_id' in df.columns:
    df['id'] = df['example_id']
elif 'id' not in df.columns:
    df['id'] = df.index  # Fallback to index if neither exists

# Save processed dataset
processed_file_path = '/kaggle/working/data/folio_h1_caroline.csv'
df_clean = df[['id', 'premises', 'conclusion', 'label', 'caroline_premises', 'caroline_conclusion', 'shuffled_premises']]
df_clean.to_csv(processed_file_path, index=False)

print(f"Processed dataset successfully saved to: {processed_file_path}")
print("\nFirst 2 processed samples:")
display(df_clean[['premises', 'caroline_premises']].head(2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.9 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
--- Step 1: Downloading and Saving RAW FOLIO Dataset ---
Raw dataset successfully saved to: /kaggle/working/data/folio_raw_original.jsonl
Loaded 204 samples into DataFrame.

--- Step 2: Applying Caroline Filter & Shuffling (H1 Preparation) ---
Processed dataset successfully saved to: /kaggle/working/data/folio_h1_caroline.csv

First 2 processed samples:


,premises,caroline_premises
0,If people perform in school talent shows often...,If people perform in school talent shows often...
1,If people perform in school talent shows often...,If people perform in school talent shows often...
